# Lab 00: How to check your lab environment

**CPU | API key: `OPENAI_API_KEY` (five short requests in Parts E to G) | Run first**

## Goal

Check in one run that your setup can run every lab in this repository. Set up first with this lab's README (your own
machine with `uv` or `pip`, GitHub Codespaces, an Ubuntu VM or Colab), then run every cell from the top. When the
last cell shows ✅ on every line, you are ready for Lab 01.

## What you will learn

- What the first two cells of every lab do, on every route.
- Where the model names, the API address and your key come from, and which value wins.
- How to read the stamp that every result in these labs carries.
- How to tell a blocked network from a key problem, and fix a failed check with the troubleshooting table.

**Run this first:** run this notebook on the machine you will use for the labs. If a line in the last cell shows ❌,
the troubleshooting table at the end says what to do.

## The first two cells

Every lab in this repository opens with the same two cells.

**Setup** finds the repository folder: it looks in the notebook's folder and in each folder above it for
`models.env`, and adds that folder to `sys.path`, so the labs can import from `shared/`, where they keep the code they
reuse. On Colab it first copies the repository into the session and installs the packages; everywhere else you
installed them once with the README's steps, so it installs nothing.

**Config** reads the settings and looks for your key. It prints the settings that decide every result, and where it
found the key, never the key itself.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


Each setting can come from three places, and a later one wins:

1. `models.env` in the repository folder: the shipped values, the same for everyone who uses this repository.
2. Your `.env` file, when it sets the same name: your machine only.
3. An environment variable with the same name, set by your system, a codespace or a terminal.

To switch every lab to another model, change one line in `models.env`; to try one on your machine only, put the line
in your `.env`. No notebook names a model in its code.

The lines above the key are the **stamp**. Every result a lab saves and every table it prints carries it, so when two
people compare numbers they can see whether the same models on the same endpoint produced them. The key is looked
for in the environment (where a codespace puts its secrets), then in `.env`, then in Colab Secrets. If the key line
says `not found`, add the key now (README, step 4 of your route) and run Config again.

The checks follow, one cell each. Each one adds its name to `passed` when it passes, and the last cell reads it.

---

## Part A: Check Python and the packages

The saved outputs in every lab come from the versions in `uv.lock`; `requirements.txt` gives the range of versions
that work, for `pip` and Colab. This cell checks the Python version and that each installed package is inside its
range, in the environment the notebook runs in.

In [3]:
import platform
from importlib.metadata import version
from packaging.requirements import Requirement

passed = set()                                     # each check adds its name here when it passes
requirements = [Requirement(line) for line in (ROOT / "requirements.txt").read_text().splitlines()
                if line and not line.startswith("#")]
outside = []
for r in requirements:
    installed = version(r.name)
    print(f"{r.name:26}{installed:10}{r.specifier}")
    if not r.specifier.contains(installed, prereleases=True):
        outside.append(r.name)
print(f"\nPython {platform.python_version()} | outside their range: {outside or 'none'}")
if (3, 11) <= sys.version_info[:2] <= (3, 14) and not outside:
    passed.add("A  Python and the packages")

openai                    3.35.1    <4,>=2.54
numpy                     2.5.4     <3,>=2.0
tiktoken                  0.14.0    <1,>=0.8
python-dotenv             1.2.4     <2,>=1.0
bm25s                     0.3.13    <0.4,>=0.3
langchain                 1.4.6     <2,>=1.4
langchain-openai          1.7.2     <2,>=1.7
langchain-text-splitters  1.1.3     <2,>=1.1
pypdf                     6.21.2    <7,>=6
pytest                    9.1.1     <10,>=8

Python 3.13.5 | outside their range: none


A pass is Python 3.11 to 3.14 and `outside their range: none`. A `pip` install can show newer versions inside each
range; that is fine.

---

## Part B: Check the data files

Every lab reads the same frozen data in `data/`, and many results depend on its exact text. `data/SHA256SUMS.txt`
holds a SHA-256 hash of each file, a fingerprint of its bytes that changes completely when one byte changes.

In [4]:
import hashlib

sums = [line.split("  ") for line in (ROOT / "data" / "SHA256SUMS.txt").read_text().splitlines()]  # hash, two spaces, name
changed = [name for expected, name in sums
           if hashlib.sha256((ROOT / "data" / name).read_bytes()).hexdigest() != expected]
print(f"{len(sums) - len(changed)} of {len(sums)} files match | changed: {changed or 'none'}")
if not changed:
    passed.add("B  data files unchanged")

5 of 5 files match | changed: none


A pass is `5 of 5 files match`. Saving a data file from an editor can be enough to change its hash, which is why the
labs only read `data/`.

---

## Part C: Check that you can reach the endpoint

This cell tests the network alone: one request without your key to each endpoint's `/models` address. Any HTTP
status, even `401` (no key was sent), means the network lets the labs through; no answer stops the cell with a
`URLError`. Part E then tests the key, so a network problem and a key problem never look alike.

In [5]:
import urllib.error
import urllib.request
from shared.settings import chat_url, embed_url

for url in dict.fromkeys([chat_url(settings), embed_url(settings)]):   # each address once
    try:
        status = urllib.request.urlopen(url + "/models", timeout=15).status
    except urllib.error.HTTPError as error:        # the server answered, with an error status
        status = error.code
    print(url, "answered: HTTP", status)
passed.add("C  endpoint reachable")

https://api.openai.com/v1 answered: HTTP 401


A pass is an answer from each endpoint, such as `https://api.openai.com/v1 answered: HTTP 401`.

---

## Part D: Check the tokenizer

Some labs count a prompt's tokens before they send it, with `tiktoken`. The first time it uses an encoding, it
downloads it from `openaipublic.blob.core.windows.net`, a second address your network must allow.

In [6]:
import tiktoken

question = "In what year was Nikola Tesla born?"           # one of the 30 development questions
encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])
print(f"{settings['TOKEN_ENCODING']}: {len(encoding.encode(question))} tokens")
passed.add("D  tokenizer loads")

o200k_base: 8 tokens


A pass is a token count: 8 tokens for the question with `o200k_base`.

---

## Part E: Check the key with one embedding

Now the key: one request to the embedding model, which turns the question into a list of numbers. Lab 01 is built
on these.

⚠️ Parts E to G send five short requests with your key; together they cost a small fraction of a US cent.

In [7]:
from shared.settings import embed_client

response = embed_client(settings).embeddings.create(model=settings["EMBED_MODEL"], input=question)
print(len(response.data[0].embedding), "numbers |", response.usage.total_tokens, "tokens billed")
passed.add("E  key works (one embedding)")

1536 numbers | 8 tokens billed


A pass is a list of numbers: 1536 with `text-embedding-3-small`. An error here after a pass in Part C is the key or
the account, not the network.

---

## Part F: Check the chat models

One question to each chat model the labs use: `CHAT_MODEL` writes the answers, `JUDGE_MODEL` grades them (Lab 07) and
`RERANK_MODEL` reorders search results (Lab 08). `chat_options()` adds the model name and either `temperature` or
`reasoning_effort`, as `REASONING_EFFORT` says; Lab 02 shows how.

In [8]:
from shared.settings import chat_client, chat_options

for name in ("CHAT_MODEL", "JUDGE_MODEL", "RERANK_MODEL"):
    reply = chat_client(settings).chat.completions.create(
        **chat_options(name, settings), messages=[{"role": "user", "content": question + " Answer with the year only."}])
    print(f"{name:14}{reply.model:26}{reply.choices[0].message.content}")
passed.add("F  chat models answer")

CHAT_MODEL    gpt-4o-mini-2024-07-18    1856


JUDGE_MODEL   gpt-4o-mini-2024-07-18    1856


RERANK_MODEL  gpt-4o-mini-2024-07-18    1856


A pass is a reply from each model, in any wording. The second column is the model that answered: `gpt-4o-mini` in
`models.env` is an alias, and OpenAI reports the dated version behind it.

---

## Part G: Check JSON replies

From Lab 02 on, the labs ask for answers as JSON that follows a schema and check every reply in code. `JSON_MODE` in
the settings says how they ask: `json_schema` makes the endpoint follow a strict schema, `json_object` only promises
valid JSON, for servers without strict schemas. This cell asks the way the labs do, with `answer()` from
`shared/answer.py` (Lab 02 builds it), which stops with an error when a reply has the wrong shape.

In [9]:
from shared.answer import answer

print(answer(question, ["[T1] Nikola Tesla was born on 10 July 1856."], settings))
passed.add("G  JSON replies have the right shape")

{'answer': '1856', 'claims': [{'claim': 'Nikola Tesla was born in 1856', 'source_id': 'T1', 'quote': 'Nikola Tesla was born on 10 July 1856.'}]}


A pass is a reply with an `answer` and a list of `claims`, each with a `source_id` and a `quote`.

---

## Check that you are ready

One line per check. A check that stopped with an error never added its name, so it shows ❌. If a line fails, copy
the whole output when you ask for help: the first line says where the notebook ran.

In [10]:
from datetime import date

CHECKS = ["A  Python and the packages", "B  data files unchanged", "C  endpoint reachable", "D  tokenizer loads",
          "E  key works (one embedding)", "F  chat models answer", "G  JSON replies have the right shape"]
print("Run on", date.today().isoformat(), "|", "Colab" if IN_COLAB else platform.platform(terse=True),
      "| Python", platform.python_version())
for check in CHECKS:
    print("✅" if check in passed else "❌", check)

Run on 2026-10-24 | macOS-15.7.9 | Python 3.13.5
✅ A  Python and the packages
✅ B  data files unchanged
✅ C  endpoint reachable
✅ D  tokenizer loads
✅ E  key works (one embedding)
✅ F  chat models answer
✅ G  JSON replies have the right shape


Every line passes: you are ready for Lab 01. A ❌ names the part to look at; the table below lists the usual causes.

### Checkpoint

| Check | You should see | Our run |
|---|---|---|
| Config | the stamp, and `OPENAI_API_KEY found in ...` | `gpt-4o-mini` and `text-embedding-3-small` at `https://api.openai.com/v1`; key found in `.env` |
| A | Python 3.11 to 3.14; outside their range: none | Python 3.13.5; outside their range: none |
| B | 5 of 5 files match | 5 of 5; changed: none |
| C | an HTTP answer from each endpoint | `https://api.openai.com/v1 answered: HTTP 401` |
| D | a token count | 8 tokens |
| E | 1536 numbers | 1536 numbers, 8 tokens billed |
| F | a reply from each chat model | `1856` from `gpt-4o-mini-2024-07-18`, three times |
| G | an answer with its claims | *1856*, one claim citing `T1` |

The outputs saved here come from a run with `uv`. Every check also passes with `pip`, and on a reasoning model with
`REASONING_EFFORT=none` (see the README's section on models).

## What you can do now

- Open any lab in this repository on your route; its first two cells are the ones you ran here.
- Change the model for every lab (`models.env`) or on your machine only (`.env`).
- Read a stamp and say which models and endpoint produced a result.
- Tell from Parts C and E whether a failure is the network or the key, and fix a failed check with the table below.

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| Setup stops with `StopIteration` | the notebook is not inside the repository folder | open it from the folder you cloned or unzipped, where `models.env` sits |
| Config stops with `ModuleNotFoundError: No module named 'shared'` | Setup did not run first | run Setup, then Config |
| Config stops with `ModuleNotFoundError: No module named 'dotenv'`, or Part A with `PackageNotFoundError: No package metadata was found for ...` | the notebook runs on a kernel without the packages | choose the `.venv` kernel (README, step 5 of your route) |
| Part A shows a Python below 3.11 or above 3.14 | Ubuntu 22.04's own Python is 3.10 | use `uv` (README, route A), which installs Python 3.13 for the repository |
| Part A lists a package outside its range | an older or newer copy in the environment | with `uv`: `uv sync`; with `pip`: `pip install -r requirements.txt` in the activated environment. Then restart the kernel and run from the top |
| Part B lists a changed file, or stops with `FileNotFoundError` | the file was edited, saved by an editor or deleted | `git checkout -- data/<file name>`, or download the file again from the repository |
| Config shows a model or endpoint you did not expect | a line in your `.env` or an environment variable overrides `models.env` | correct or delete that line, then run Config again |
| Part C: `URLError: <urlopen error [Errno 61] Connection refused>` or `[Errno 8] nodename nor servname provided, or not known` (Linux: `[Errno 111]` or `Name or service not known`) | a wrong `OPENAI_BASE_URL`, no connection, or a firewall | check `OPENAI_BASE_URL` in your `.env` (empty means OpenAI); try another network; ask IT to allow the address |
| Part C: `URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] ...>` | a company proxy that inspects web traffic | use another network, a codespace or Colab, or ask IT how Python should trust the company's certificate |
| Part D: `ProxyError` or `ConnectionError` for `openaipublic.blob.core.windows.net` | the network blocks the tokenizer's one-time download | ask IT to allow the address, or copy the cache folder from a machine where Part D passed and set the environment variable `TIKTOKEN_CACHE_DIR` to it |
| Part E: `OpenAIError: Missing credentials` | no key in the environment, `.env` or Colab Secrets | add it (README, step 4 of your route; Codespaces: add the secret and restart the codespace), then run Config again |
| Part E: `AuthenticationError: Error code: 401` with `Incorrect API key provided` | the key is mistyped, revoked or expired | paste the key again (no quotes, no spaces) or create a new one; for a key you were given, ask whoever gave it to you |
| Part F: `NotFoundError: Error code: 404` with ``The model `...` does not exist or you do not have access to it`` | a misspelt model name, or your account cannot use that model | check the model lines in `models.env` and your `.env` |
| Part F: `BadRequestError: Error code: 400` with `Unrecognized request argument supplied: reasoning_effort` | `REASONING_EFFORT` is set, and one of the chat models has no such setting | set all three chat models to the reasoning model, or empty `REASONING_EFFORT` |
| Part F: `BadRequestError: Error code: 400` with `Unsupported value: 'temperature' does not support 0 with this model` | a reasoning model such as `gpt-6-luna` with `REASONING_EFFORT` empty | set `REASONING_EFFORT=none` |
| Parts E to G: `RateLimitError: Error code: 429` | a shared key reached its limit, or the account has no credit | wait a minute and run the cell again; if it repeats, ask whoever gave you the key |
| Part G fails while Part F passes | the endpoint does not support strict JSON schemas | set `JSON_MODE=json_object` in your `.env`, then run Config and Part G again |

The README has a second table for problems during setup, before the notebook opens.

## Resources

- [OpenAI: embeddings](https://developers.openai.com/api/docs/guides/embeddings): the call in Part E and the
  embedding models you can set in `models.env`
- [OpenAI: structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs): `json_schema`
  against `json_object`, the choice in Part G
- [OpenAI: deprecations](https://developers.openai.com/api/docs/deprecations): shutdown dates for models such as
  `gpt-4o-mini`
- [tiktoken](https://github.com/openai/tiktoken): the tokenizer in Part D

Next: [Lab 01: How to search documents with embeddings](../01_Search_With_Embeddings/README.md). Back to
[the lab list](../README.md).